# 🎬 Video2X trên Kaggle (AI Video Super-Resolution & Upscaling)

Notebook này cung cấp giải pháp nâng cấp chất lượng video bằng trí tuệ nhân tạo (AI Video Upscaling) dựa trên **[Video2X v6.2.0](https://github.com/k4yt3x/video2x)** (phiên bản C++ native tối ưu hiệu năng), tận dụng phần cứng GPU miễn phí (**NVIDIA Tesla T4 x2 / P100**) trên **Kaggle**.

---

### ⚠️ Cài đặt bắt buộc trên Kaggle trước khi chạy:

> [!IMPORTANT]
> **1. Bật GPU:** Tại bảng điều khiển bên phải (**Notebook Settings**) ➡️ mục **Accelerator** ➡️ chọn **GPU T4 x2** (hoặc **GPU P100**).
> 
> **2. Bật Internet:** Tại bảng **Settings** ➡️ gạt toggle **Internet** sang **ON** (bắt buộc để tải Video2X, FFmpeg 7 và mô hình AI).

---

### 🚀 Quy trình thực hiện:
```
[Bước 0: Kiểm tra GPU] ➡️ [Bước 1: Cài đặt môi trường] ➡️ [Bước 2: Chọn video đầu vào] ➡️ [Bước 3: Cấu hình & Upscale] ➡️ [Bước 4: Tải kết quả]
```


## 🖥️ Bước 0: Kiểm tra phần cứng GPU

Chạy lệnh `nvidia-smi` bên dưới để:
* Xác nhận Kaggle đã cấp phát GPU NVIDIA và driver đang hoạt động.
* Nhận diện phiên bản **Driver Version** (ví dụ: `535`, `550`, `580`) để script ở **Bước 1** tự động cài đặt thư viện đồ họa Vulkan tương thích.


In [1]:
# Kiểm tra thông tin GPU được cấp phát
!nvidia-smi


Thu Nov  5 10:23:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📦 Bước 1: Cài đặt Video2X v6.2.0, FFmpeg 7 & Kích hoạt Vulkan NVIDIA

Cell bên dưới sẽ tự động tải và cấu hình toàn bộ môi trường cần thiết:
1. **FFmpeg 7 (`libavcodec.so.61`)**: Cài đặt thông qua PPA `ubuntuhandbook1/ffmpeg7` (bắt buộc vì Video2X v6.2.0 được biên dịch với FFmpeg 7).
2. **NVIDIA Vulkan Driver (`libnvidia-gl-*`)**: Tự động nhận diện phiên bản driver của máy và cài đặt thư viện đồ họa Vulkan NVIDIA vào container (khắc phục triệt để lỗi `Failed to create Vulkan instance`).
3. **Video2X v6.2.0**: Tải và cài đặt bản binary Linux C++ native tốc độ cao.
4. **Cấu hình Vulkan ICD**: Thiết lập biến môi trường chỉ định Vulkan chạy trực tiếp trên GPU NVIDIA.

> [!NOTE]
> Quá trình cài đặt chỉ mất khoảng **1 - 2 phút** và chỉ cần chạy **một lần duy nhất** cho mỗi phiên làm việc (session).


In [2]:
import os
import re
import json
import subprocess
from pathlib import Path

print("⏳ [1/5] Thêm PPA FFmpeg 7 và cập nhật danh sách gói...")
!apt-get update -yqq
!apt-get install -yqq software-properties-common
!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg7
!apt-get update -yqq
!apt-get install -yqq --no-install-recommends libvulkan1 libvulkan-dev vulkan-tools ffmpeg mesa-vulkan-drivers

print("\n⏳ [2/5] Nhận diện và cài đặt thư viện Vulkan NVIDIA (libnvidia-gl)...")
# Tự động nhận diện phiên bản driver NVIDIA từ nvidia-smi
driver_ver = '535'
try:
    smi_out = subprocess.check_output('nvidia-smi', shell=True).decode()
    m = re.search(r'Driver Version:\s*(\d+)', smi_out)
    if m:
        driver_ver = m.group(1)
        print(f'🔍 Phiên bản driver phát hiện: {driver_ver}')
except Exception as e:
    print(f'⚠️ Không đọc được nvidia-smi, sử dụng mặc định 535: {e}')

# Cài đặt thư viện đồ họa user-space NVIDIA Vulkan (sửa lỗi Failed to create Vulkan instance)
ret = os.system(f'apt-get install -yqq --no-install-recommends libnvidia-gl-{driver_ver}')
if ret != 0:
    print(f'⚠️ libnvidia-gl-{driver_ver} không có sẵn, thử cài libnvidia-gl-535...')
    os.system('apt-get install -yqq --no-install-recommends libnvidia-gl-535')

print("\n⏳ [3/5] Tải và cài đặt Video2X v6.2.0...")
deb_file = "/tmp/video2x-linux-ubuntu2204-amd64.deb"
!wget -q --show-progress -O {deb_file} https://github.com/k4yt3x/video2x/releases/download/6.2.0/video2x-linux-ubuntu2204-amd64.deb
!apt-get install -yqq {deb_file} || (dpkg -i {deb_file} && apt-get install -fyqq)

print("\n⏳ [4/5] Cấu hình Vulkan ICD cho NVIDIA GPU...")
icd_dir = Path("/usr/share/vulkan/icd.d")
icd_dir.mkdir(parents=True, exist_ok=True)
nvidia_icd_path = icd_dir / "nvidia_icd.json"

# Tạo file ICD trỏ đến thư viện NVIDIA
if not nvidia_icd_path.exists():
    with open(nvidia_icd_path, "w") as f:
        json.dump({
            "file_format_version": "1.0.0",
            "ICD": {
                "library_path": "libGLX_nvidia.so.0",
                "api_version": "1.3.0"
            }
        }, f, indent=4)

!cp -f /usr/share/vulkan/icd.d/nvidia_icd.json /etc/vulkan/icd.d/ 2>/dev/null || true

# Thiết lập biến môi trường
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"
!ldconfig

print("\n⏳ [5/5] Kiểm tra thiết bị Vulkan và Video2X...")
!vulkaninfo --summary 2>/dev/null | grep -E "deviceName|driverName" || !vulkaninfo --summary || true
!video2x --version

print("\n✅ Cài đặt hoàn tất! GPU NVIDIA đã sẵn sàng cho Video2X.")


⏳ [1/5] Thêm PPA FFmpeg 7 và cập nhật danh sách gói...
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Repository: 'Types: deb
URIs: https://ppa.launchpadcontent.net/ubuntuhandbook1/ffmpeg7/ubuntu/
Suites: noble
Components: main
'
Description:
unofficial build for FFmpeg 7 for Ubuntu 22.04 | 24.04, backport from Debian's deb.multimedia.org repository

If the packages here are helpful, you may buy me a coffee:

         https://ko-fi.com/ubuntuhandbook1
More info: https://launchpad.net/~ubuntuhandbook1/+archive/ubuntu/ffmpeg7
Adding repository.
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linu

## 🎬 Bước 2: Chọn video đầu vào

Chỉ định đường dẫn file video bạn muốn xử lý vào biến `selected_input_file` bên dưới.

### 💡 Hướng dẫn nguồn video:
* **Từ Kaggle Dataset (Khuyên dùng):** Nhấn **+ Add Input** (góc trên bên phải) ➡️ Thêm Dataset chứa video hoặc tải video lên dưới dạng Dataset mới. Đường dẫn sẽ có dạng:  
  `/kaggle/input/<tên-dataset>/<tên-video>.mp4`
* **Từ thư mục làm việc:** Nếu bạn đã tải file vào Kaggle (qua `wget`, `curl` hoặc upload widget), đường dẫn thường có dạng:  
  `/kaggle/working/input/<tên-video>.mp4`

> [!TIP]
> Đối với video dài, bạn nên cắt thử một đoạn clip ngắn (khoảng 5 - 15 giây) để chạy thử nghiệm trước nhằm chọn được bộ lọc (`PROCESSOR`), mô hình (`MODEL`) và độ nén (`CRF`) ưng ý nhất.


In [3]:

import pathlib
import ipywidgets as widgets
from IPython.display import display

# Nhập đường dẫn video input
selected_input_file = "/kaggle/input/datasets/hoacong/video-test/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0.mp4"

print("Đường dẫn video đầu vào:", selected_input_file)

Đường dẫn video đầu vào: /kaggle/input/datasets/hoacong/video-test/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0.mp4


## ⚙️ Bước 3: Cấu hình và Thực thi Video Upscale

Tùy chỉnh các thông số trong khối code bên dưới theo nhu cầu của bạn trước khi bấm chạy.

---

### 📊 Bảng so sánh các mô hình AI:

| Bộ xử lý (`PROCESSOR`) | Mô hình (`REALESRGAN_MODEL`) | Tỷ lệ Scale | Thích hợp cho | Đánh giá |
| :--- | :--- | :---: | :--- | :--- |
| **`realesrgan`** | `realesr-animevideov3` | 2x, 3x, 4x | Hoạt hình, Anime, Clip 2D | ⚡ Tốc độ nhanh nhất, chuyển động mượt, rất khuyên dùng |
| **`realesrgan`** | `realesrgan-plus-anime` | 4x | Hoạt hình, Anime nét cao | 🎨 Độ sắc nét và chi tiết cực cao (chạy lâu hơn animevideov3) |
| **`realesrgan`** | `realesrgan-plus` | 4x | Người thật, phong cảnh, đời thực | 📷 Khử nhiễu, tái tạo chi tiết ảnh chụp thực tế tốt |
| **`libplacebo`** | `anime4k-v4-a / b / c` | Tùy biến | Anime độ phân giải từ 480p - 1080p | ⚡ Dùng GLSL shader thời gian thực, tốc độ siêu tốc |

---

### 🎛️ Tùy chỉnh thông số nén video (FFmpeg Encoder):
* **`CRF` (Chất lượng hình ảnh)**:
  * `16`: Chất lượng cực cao (gần như lossless), chi tiết giữ nguyên tối đa nhưng file xuất ra nặng hơn.
  * `18 - 20`: **Mức cân bằng lý tưởng** giữa chất lượng hình ảnh sắc nét và dung lượng file hợp lý.
  * `22 - 24`: Dung lượng nhẹ, phù hợp nếu muốn tiết kiệm bộ nhớ hoặc video dài.
* **`PRESET`**: `slow` (khuyên dùng để nén chất lượng tốt nhất) hoặc `medium`, `fast`.
* **`PIXFMT`**: Nên giữ `yuv420p` để đảm bảo video phát được trên mọi thiết bị, điện thoại và trình duyệt web.

> [!NOTE]
> **Về thông báo `Exit code: 139 / Segmentation fault` lúc kết thúc:**  
> Nếu bạn thấy xuất hiện thông báo này sau khi log đã in `Video processed successfully` và `Output written to...`, bạn hoàn toàn yên tâm vì **video đã hoàn thành 100% và file đã được ghi trọn vẹn**. Đây chỉ là hiện tượng driver Vulkan giải phóng bộ nhớ lúc thoát ứng dụng, không hề ảnh hưởng đến video kết quả.


In [4]:
import os
import time
import pathlib
import subprocess

# ==========================================================
# ⚙️ TÙY CHỈNH THÔNG SỐ UPSCALING TẠI ĐÂY
# ==========================================================
PROCESSOR = "realesrgan"            # "realesrgan" hoặc "libplacebo"

# Cấu hình Real-ESRGAN
REALESRGAN_MODEL = "realesrgan-plus-anime"  # "realesr-animevideov3", "realesrgan-plus-anime", "realesrgan-plus"
SCALE = 4                                  # 2, 3, hoặc 4 (Model animevideov3 hỗ trợ 2, 3, 4; model khác chỉ 4)

# Cấu hình libplacebo (Anime4K)
SHADER = "anime4k-v4-a"             # "anime4k-v4-a", "anime4k-v4-b", "anime4k-v4-c", "anime4k-v4.1-gan"
TARGET_WIDTH = 1920                 # Đặt 0 nếu muốn tự tính toán
TARGET_HEIGHT = 1080                # Đặt 0 nếu muốn tự tính toán

# Cấu hình FFmpeg Encoder
CODEC = "libx264"                   # "libx264" (CPU x264 chất lượng cao) hoặc "h264_nvenc" (NVIDIA Hardware)
PRESET = "slow"                    # "ultrafast", "fast", "medium", "slow", "veryslow"
CRF = 14                            # 14 - 23 (số càng thấp chất lượng càng cao)
PIXFMT = "yuv420p"                     # "auto", "yuv420p"
BITRATE = 0                         # 0 để dùng CRF
LOGLEVEL = "info"                   # "info", "debug", "warn", "error"

# Thư mục lưu kết quả
OUTPUT_DIR = pathlib.Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ==========================================================

# Kiểm tra đầu vào
if 'selected_input_file' not in globals() or selected_input_file is None or not pathlib.Path(selected_input_file).exists():
    raise ValueError("❌ Bạn chưa chọn video đầu vào! Vui lòng hoàn thành Bước 2 trước.")

input_path = pathlib.Path(selected_input_file)
output_path = OUTPUT_DIR / f"{input_path.stem}_upscaled{input_path.suffix}"

# Xóa file cũ nếu đã tồn tại để tránh xung đột
if output_path.exists():
    output_path.unlink()

print(f"🎬 Video nguồn: {input_path}")
print(f"💾 Video đích  : {output_path}")
print(f"🚀 Processor  : {PROCESSOR} | Scale: {SCALE}x")

# Xây dựng lệnh video2x
cmd = [
    "video2x",
    "--input", f'"{input_path}"',
    "--output", f'"{output_path}"',
    "--processor", PROCESSOR,
    "--codec", CODEC,
    "--log-level", LOGLEVEL,
    "-e", f"preset={PRESET}",
    "-e", f"crf={CRF}"
]

if BITRATE > 0:
    cmd.extend(["--bit-rate", str(BITRATE)])

if PIXFMT != "auto":
    cmd.extend(["--pix-fmt", PIXFMT])

if PROCESSOR == "realesrgan":
    cmd.extend([
        "--realesrgan-model", REALESRGAN_MODEL,
        "--scaling-factor", str(SCALE)
    ])
elif PROCESSOR == "libplacebo":
    cmd.extend(["--libplacebo-shader", f'"{SHADER}"'])
    if TARGET_WIDTH > 0:
        cmd.extend(["--width", str(TARGET_WIDTH)])
    if TARGET_HEIGHT > 0:
        cmd.extend(["--height", str(TARGET_HEIGHT)])

full_cmd = " ".join(cmd)
print("-" * 60)
print(f"Executing: {full_cmd}")
print("-" * 60)

start_time = time.time()
p = subprocess.Popen(full_cmd, shell=True)
p.communicate()
elapsed = time.time() - start_time

# Kiểm tra kết quả (Lưu ý: Video2X có thể thoát mã 139 lúc giải phóng Vulkan nhưng file đã ghi xong)
if output_path.exists() and output_path.stat().st_size > 1024:
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print("=" * 60)
    if p.returncode in (0, 139, -11):
        print(f"🎉 XỬ LÝ THÀNH CÔNG trong {elapsed:.1f} giây!")
        if p.returncode in (139, -11):
            print("ℹ️ Ghi chú: Mã thoát 139 (SIGSEGV) xuất hiện khi hủy Vulkan driver lúc đóng ứng dụng, video đã hoàn tất 100%.")
    else:
        print(f"⚠️ Video đã xuất ra ({size_mb:.2f} MB), nhưng chương trình trả về mã thoát: {p.returncode}")
    print(f"📁 File kết quả: {output_path} ({size_mb:.2f} MB)")
    print("=" * 60)
else:
    print("=" * 60)
    print(f"❌ Đã xảy ra lỗi khi xử lý! (Exit code: {p.returncode})")
    print("=" * 60)


🎬 Video nguồn: /kaggle/input/datasets/hoacong/video-test/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0.mp4
💾 Video đích  : /kaggle/working/output/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0_upscaled.mp4
🚀 Processor  : realesrgan | Scale: 4x
------------------------------------------------------------
Executing: video2x --input "/kaggle/input/datasets/hoacong/video-test/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0.mp4" --output "/kaggle/working/output/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0_upscaled.mp4" --processor realesrgan --codec libx264 --log-level info -e preset=slow -e crf=14 --pix-fmt yuv420p --realesrgan-model realesrgan-plus-anime --scaling-factor 4
------------------------------------------------------------
[2026-11-05 10:25:17.331] [info] Video2X version 6.2.0
[2026-11-05 10:25:1

[libx264 @ 0x7f89e8007540] using cpu capabilities: MMX2 SSE2Fast SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2 AVX512
[libx264 @ 0x7f89e8007540] profile High, level 6.0, 4:2:0, 8-bit
[libx264 @ 0x7f89e8007540] 264 - core 164 r3108 31e19f9 - H.264/MPEG-4 AVC codec - Copyleft 2003-2023 - http://www.videolan.org/x264.html - options: cabac=1 ref=5 deblock=1:0:0 analyse=0x3:0x113 me=hex subme=8 psy=1 psy_rd=1.00:0.00 mixed_ref=1 me_range=16 chroma_me=1 trellis=2 8x8dct=1 cqm=0 deadzone=21,11 fast_pskip=1 chroma_qp_offset=-2 threads=6 lookahead_threads=1 sliced_threads=0 nr=0 decimate=1 interlaced=0 bluray_compat=0 constrained_intra=0 bframes=3 b_pyramid=2 b_adapt=1 b_bias=0 direct=3 weightb=1 open_gop=0 weightp=2 keyint=250 keyint_min=24 scenecut=40 intra_refresh=0 rc_lookahead=50 rc=crf mbtree=1 crf=14.0 qcomp=0.60 qpmin=0 qpmax=69 qpstep=4 ip_ratio=1.40 aq=1:1.00
[0 Tesla T4]  queueC=2[8]  queueG=0[16]  queueT=1[2]
[0 Tesla T4]  bugsbn1=0  bugbilz=0  bugcopc=0  bugihfa=0
[0 Tesla T4]  fp16-p/s/a=1/1/1 

frame=158/158 (100.00%); fps=0.31; elapsed=00:08:33; remaining=00:00:00

[libx264 @ 0x7f89e8007540] frame I:1     Avg QP:12.43  size:1453560
[libx264 @ 0x7f89e8007540] frame P:40    Avg QP:12.26  size:909372
[libx264 @ 0x7f89e8007540] frame B:117   Avg QP:14.68  size:334635
[libx264 @ 0x7f89e8007540] consecutive B-frames:  1.3%  0.0%  0.0% 98.7%
[libx264 @ 0x7f89e8007540] mb I  I16..4: 15.4% 48.3% 36.4%
[libx264 @ 0x7f89e8007540] mb P  I16..4:  6.5% 12.2%  3.9%  P16..4: 31.1% 26.0% 14.0%  0.0%  0.0%    skip: 6.3%
[libx264 @ 0x7f89e8007540] mb B  I16..4:  0.7%  0.8%  0.3%  B16..8: 38.7% 15.4%  3.9%  direct:12.4%  skip:27.8%  L0:46.8% L1:39.6% BI:13.6%
[libx264 @ 0x7f89e8007540] 8x8 transform intra:52.0% inter:44.3%
[libx264 @ 0x7f89e8007540] direct mvs  spatial:96.6% temporal:3.4%
[libx264 @ 0x7f89e8007540] coded y,uvDC,uvAC intra: 49.1% 75.3% 63.5% inter: 25.3% 39.0% 8.6%
[libx264 @ 0x7f89e8007540] i16 v,h,dc,p: 65% 20%  6% 10%
[libx264 @ 0x7f89e8007540] i8 v,h,dc,ddl,ddr,vr,hd,vl,hu: 11% 21% 26%  6%  7%  6%  9%  6%  9%
[libx264 @ 0x7f89e8007540] i4 v,h,dc,


[2026-11-05 10:33:50.812] [info] Video processed successfully
====== Video2X Processing summary ======
Video file processed: /kaggle/input/datasets/hoacong/video-test/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0.mp4
Total frames processed: 158
Total time taken: 00:08:33
Average processing speed: 0.31 FPS
Output written to: /kaggle/working/output/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0_upscaled.mp4
🎉 XỬ LÝ THÀNH CÔNG trong 514.5 giây!
ℹ️ Ghi chú: Mã thoát 139 (SIGSEGV) xuất hiện khi hủy Vulkan driver lúc đóng ứng dụng, video đã hoàn tất 100%.
📁 File kết quả: /kaggle/working/output/hailuo-03_Animate_only_these_five_elements_in_the_original_image_1._Gentle_wind_A_soft_bre-0_upscaled.mp4 (73.58 MB)


Segmentation fault (core dumped)


## ⏩ Bước 4 (Tùy chọn): Tăng FPS với mô hình RIFE

Video2X hỗ trợ mô hình RIFE (Real-Time Intermediate Flow Estimation) để nhân số khung hình (ví dụ nhân đôi từ 24fps/30fps lên 60fps), giúp chuyển động mượt mà hơn đáng kể.

In [ ]:

import time
import pathlib
import subprocess
import shutil

# Cấu hình RIFE
RIFE_MODEL = "rife-v4.6" # "rife-v4.6", "rife-anime", "rife-v4", "rife-UHD", "rife-v2.4"
FRAME_RATE_MUL = 2 # Hệ số nhân FPS (2: nhân đôi, 4: nhân 4)
CODEC = "libx264"
PRESET = "slow"
CRF = 16

OUTPUT_DIR = pathlib.Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra video đầu vào
if (
    'selected_input_file' not in globals()
    or not pathlib.Path(selected_input_file).is_file()
):
    raise ValueError("❌ Chưa chọn video đầu vào!")

# Kiểm tra Video2X
if shutil.which("video2x") is None:
    raise RuntimeError("❌ Không tìm thấy video2x!")

input_path = pathlib.Path(selected_input_file)
output_rife = OUTPUT_DIR / (
    f"{input_path.stem}_rife_{FRAME_RATE_MUL}x{input_path.suffix}"
)

# Xóa kết quả cũ
if output_rife.exists():
    output_rife.unlink()

# Tạo lệnh RIFE
rife_cmd = [
    "video2x",
    "--input", str(input_path),
    "--output", str(output_rife),
    "--processor", "rife",
    "--rife-model", RIFE_MODEL,
    "--frame-rate-mul", str(FRAME_RATE_MUL),
    "--codec", CODEC,
    "--log-level", "info",
    "-e", f"preset={PRESET}",
    "-e", f"crf={CRF}"
]

print("🚀 Đang chạy RIFE frame interpolation:")
print(subprocess.list2cmdline(rife_cmd))
print("-" * 60)

start_t = time.time()

try:
    res = subprocess.run(rife_cmd, check=False)
    dur = time.time() - start_t

    if res.returncode == 0 and output_rife.is_file():
        mb = output_rife.stat().st_size / (1024 * 1024)

        print("=" * 60)
        print(f"🎉 Hoàn tất RIFE trong {dur:.1f} giây!")
        print(f"📁 File kết quả: {output_rife}")
        print(f"📦 Dung lượng: {mb:.2f} MB")
        print("=" * 60)

    else:
        print("=" * 60)
        print(f"⚠️ RIFE thất bại! Exit code: {res.returncode}")
        print("=" * 60)

except Exception as e:
    print(f"❌ Lỗi khi chạy RIFE: {e}")

## 📥 Bước 5: Tải video kết quả về máy

Sau khi quá trình upscale hoàn tất, file video đã được lưu hoàn chỉnh tại thư mục:
`📁 /kaggle/working/output/`

### 💻 Cách tải video về thiết bị:
1. Nhìn sang cột công cụ bên phải giao diện Kaggle ➡️ mở tab **Output**.
2. Tìm thư mục `output` ➡️ nhấp vào file video có đuôi `_upscaled.mp4`.
3. Nhấn vào biểu tượng **Ba chấm (⋮)** hoặc nút **Download** để tải video về máy tính.

> [!TIP]
> Bạn cũng có thể xem video trực tiếp hoặc tạo link tải ngay trong notebook bằng đoạn code:
> ```python
> from IPython.display import FileLink
> FileLink(str(output_path))
> ```
